<a href="https://colab.research.google.com/github/sreevarshini22/CODSOFT/blob/main/Task2/Fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

#Generate synthetic dataset
np.random.seed(42)

X, y = make_classification(
    n_samples=10_000,
    n_features=20,
    n_informative=15,
    n_redundant=3,
    n_clusters_per_class=2,
    weights=[0.975, 0.025],   # 2.5% fraud
    flip_y=0.01,
    random_state=42
)

feature_names = ['Amount', 'Hour'] + [f'V{i}' for i in range(1, 19)]
df = pd.DataFrame(X, columns=feature_names)
df['Amount'] = np.abs(df['Amount']) * 100 + 10
df['Class'] = y

print("Dataset summary")
print(f"  Total transactions : {len(df):,}")
print(f"  Legitimate         : {(df.Class == 0).sum():,}")
print(f"  Fraudulent         : {(df.Class == 1).sum():,}")
print(f"  Fraud rate         : {df.Class.mean()*100:.2f}%\n")

#Train / test split & scaling
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('Class', axis=1).values, df['Class'].values,
    test_size=0.2, random_state=42, stratify=df['Class']
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

#Models
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        max_depth=10, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model':  model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc':    roc_auc_score(y_test, y_prob),
        'ap':     average_precision_score(y_test, y_prob),
        'report': classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']),
        'cm':     confusion_matrix(y_test, y_pred),
    }

#Print results
for name, r in results.items():
    print(f"{'─'*55}")
    print(f"  {name}")
    print(f"  AUC-ROC: {r['auc']:.4f}   Avg Precision: {r['ap']:.4f}")
    print(f"{'─'*55}")
    print(r['report'])

# Plots
colors = {'Logistic Regression': '#2563EB',
          'Decision Tree':       '#6B7280',
          'Random Forest':       '#16A34A'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#Confusion matrices
for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(r['cm'], annot=True, fmt='d', ax=ax,
                cmap='Blues', cbar=False,
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}\nAUC={r["auc"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.close()

# — ROC curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax1.plot(fpr, tpr, color=colors[name], lw=2,
             label=f"{name} (AUC={r['auc']:.3f})")
ax1.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
        title='ROC Curves')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

#Precision-recall curves
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    ax2.plot(rec, prec, color=colors[name], lw=2,
             label=f"{name} (AP={r['ap']:.3f})")
ax2.set(xlabel='Recall', ylabel='Precision',
        title='Precision-Recall Curves')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150)
plt.close()

#Feature importance (Random Forest)
rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
fi.tail(10).plot(kind='barh', ax=ax, color='#2563EB')
ax.set(title='Top 10 Feature Importances (Random Forest)',
       xlabel='Mean decrease in impurity')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.close()

print("Plots saved: confusion_matrices.png, roc_pr_curves.png, feature_importance.png")


Dataset summary
  Total transactions : 10,000
  Legitimate         : 9,702
  Fraudulent         : 298
  Fraud rate         : 2.98%

───────────────────────────────────────────────────────
  Logistic Regression
  AUC-ROC: 0.7937   Avg Precision: 0.4255
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

       Legit       0.99      0.85      0.91      1940
       Fraud       0.11      0.62      0.19        60

    accuracy                           0.84      2000
   macro avg       0.55      0.73      0.55      2000
weighted avg       0.96      0.84      0.89      2000

───────────────────────────────────────────────────────
  Decision Tree
  AUC-ROC: 0.6488   Avg Precision: 0.1286
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

       Legit       0.98      0.94      0.96      1940
       Fraud       0.15      0.37      0.22        60

    accuracy                   